# Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning, ValueWarning
from itertools import product # Для генерации комбинаций

# Игнорирование предупреждений
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', ValueWarning)
warnings.simplefilter('ignore', FutureWarning)
plt.style.use('seaborn-v0_8-whitegrid')

# Functions

In [8]:
def get_stationarity(timeseries):
    d = 0
    current_series = timeseries.copy()
    if current_series.empty or len(current_series.dropna()) < 5: return 0
    try:
        result_adf = adfuller(current_series.dropna())
    except Exception: return 0
    if result_adf[1] <= 0.05: return 0 # Уже стационарен
    # print(f'Исходный ряд: ADF={result_adf[0]:.2f}, p-value={result_adf[1]:.3f}')
    while result_adf[1] > 0.05 and d < 2:
        d += 1
        current_series = current_series.diff().dropna()
        if current_series.empty or len(current_series) < 5 : return d-1 if d > 0 else 0
        try:
            result_adf = adfuller(current_series)
            # print(f'd={d}: ADF={result_adf[0]:.2f}, p-value={result_adf[1]:.3f}')
        except Exception: return d-1 if d > 0 else 0
    # if result_adf[1] <= 0.05: print(f"Ряд стационарен после {d} дифференцирования(ий).")
    # else: print(f"Ряд НЕ стационарен даже после {d} дифференцирования(ий). Используем d={d}.")
    return d

In [9]:
def generate_arima_forecast_for_combo(series_actual_price_input, 
                                      combo_dict, # Словарь с текущей комбинацией {'type':'D', 'old_new':'Y'}
                                      n_periods=3, 
                                      p_max=2, q_max=2,
                                      min_obs_for_arima=20): # Минимальное кол-во наблюдений для ARIMA
    
    combo_str_name = "_".join([f"{k}_{v}" for k, v in combo_dict.items()]) # Для логов

    if series_actual_price_input.empty or len(series_actual_price_input.dropna()) < min_obs_for_arima:
        # print(f"Пропуск комбинации {combo_str_name}: недостаточно данных ({len(series_actual_price_input.dropna())} < {min_obs_for_arima}).")
        return None 

    series_cleaned_actual = series_actual_price_input.dropna()
    if len(series_cleaned_actual) < min_obs_for_arima: # Повторная проверка после dropna
        # print(f"Пропуск комбинации {combo_str_name}: недостаточно данных после dropna ({len(series_cleaned_actual)} < {min_obs_for_arima}).")
        return None

    optimal_d = get_stationarity(series_cleaned_actual)
    best_aic = np.inf; best_order = None; best_model = None
    
    series_to_fit_main = series_cleaned_actual.copy()
    if series_to_fit_main.index.freqstr is None: # Установка частоты
        series_to_fit_main = series_to_fit_main.asfreq('AS')
        series_to_fit_main.interpolate(method='linear', limit_direction='both', inplace=True)
        series_to_fit_main.dropna(inplace=True) 
        if len(series_to_fit_main) < 10: # Проверка после установки частоты
             # print(f"Пропуск {combo_str_name}: недостаточно данных после установки частоты.")
             return None

    # Подбор ARIMA
    for p_val in range(p_max + 1):
        for q_val in range(q_max + 1):
            if p_val == 0 and q_val == 0 and optimal_d == 0: current_order = (0,0,0)
            elif p_val == 0 and q_val == 0 and optimal_d > 0: continue
            else: current_order = (p_val, optimal_d, q_val)
            try:
                temp_model = ARIMA(series_to_fit_main, order=current_order, enforce_stationarity=False, enforce_invertibility=False).fit()
                if temp_model.aic < best_aic:
                    best_aic = temp_model.aic; best_order = current_order; best_model = temp_model
            except Exception: continue
    
    if best_model is None: # Резервные модели
        default_orders_to_try = [(1, optimal_d, 0), (0, optimal_d, 1), (1, optimal_d, 1)]
        if optimal_d == 0: default_orders_to_try.insert(0, (1,0,0))
        for order_try in default_orders_to_try:
            try:
                best_model = ARIMA(series_to_fit_main, order=order_try, enforce_stationarity=False, enforce_invertibility=False).fit()
                best_order = order_try; best_aic = best_model.aic
                # print(f"  Используем резервную модель {best_order} (AIC: {best_aic:.2f}) для {combo_str_name}")
                break
            except: continue
            
    if best_model is None:
        # print(f"Не удалось подобрать модель ARIMA для {combo_str_name}. Пропуск.")
        return None

    # print(f"Лучшая модель для {combo_str_name}: ARIMA{best_order} (AIC={best_aic:.2f})")

    forecast_actual_values = best_model.forecast(steps=n_periods)
    forecast_obj = best_model.get_forecast(steps=n_periods)
    conf_int_actual_df = forecast_obj.conf_int()

    last_hist_year = series_cleaned_actual.index.max().year
    forecast_index = pd.date_range(start=f'{last_hist_year + 1}-01-01', periods=n_periods, freq='AS')
    
    forecast_series_actual = pd.Series(forecast_actual_values, index=forecast_index)
    conf_int_actual_df.index = forecast_index
    conf_int_actual_df.columns = ['lower_ci', 'upper_ci']

    # Подготовка данных для вывода в длинном формате
    historical_df = series_cleaned_actual.reset_index()
    historical_df.columns = ['year_dt', 'price_value'] # Год как datetime
    historical_df['value_type'] = 'historical'
    historical_df['lower_ci'] = np.nan
    historical_df['upper_ci'] = np.nan

    forecast_df = forecast_series_actual.reset_index()
    forecast_df.columns = ['year_dt', 'price_value'] # Год как datetime
    forecast_df['value_type'] = 'forecast'
    forecast_df = forecast_df.merge(conf_int_actual_df.reset_index(drop=True), left_index=True, right_index=True)

    result_df_long = pd.concat([historical_df, forecast_df], ignore_index=True)
    
    # Добавляем колонки с параметрами текущей комбинации
    for param_key, param_value in combo_dict.items():
        result_df_long[param_key] = param_value
        
    # Преобразуем год в числовой формат (только год)
    if pd.api.types.is_datetime64_any_dtype(result_df_long['year_dt']):
        result_df_long['year'] = result_df_long['year_dt'].dt.year
        result_df_long.drop(columns=['year_dt'], inplace=True)
        
    return result_df_long[['year'] + list(combo_dict.keys()) + ['price_value', 'value_type', 'lower_ci', 'upper_ci']]

In [10]:
def prepare_timeseries_for_combo(df_filtered_input): # Принимает уже отфильтрованный DF
    if df_filtered_input.empty: return pd.Series(dtype='float64')
        
    df_agg = df_filtered_input.groupby('year')['price'].mean()
    if df_agg.empty: return pd.Series(dtype='float64')

    df_agg.index = pd.to_datetime(df_agg.index, format='%Y')
    
    if not df_agg.empty and len(df_agg.dropna()) > 1:
        min_year_series = df_agg.index.min().year
        max_year_series = df_agg.index.max().year
        full_year_index_series = pd.date_range(start=f'{min_year_series}-01-01', 
                                               end=f'{max_year_series}-01-01', 
                                               freq='AS')
        df_agg_filled = df_agg.reindex(full_year_index_series).interpolate(method='linear', limit_direction='both')
        return df_agg_filled.dropna() 
    elif not df_agg.empty:
        return df_agg.dropna()
    return pd.Series(dtype='float64')

# Data

In [11]:
df_ts_main = pd.read_parquet('./data/pp-complete.parquet')
df_ts_main.dropna(subset=['date', 'price'], inplace=True)
df_ts_main['year'] = df_ts_main['date'].dt.year

In [12]:
combination_columns = ['type', 'old_new', 'duration', 'ppd_type']

unique_values = {}
for col in combination_columns:
    # Убедимся, что колонка существует и значения не все NaN
    if col in df_ts_main.columns and df_ts_main[col].notna().any():
        unique_vals = df_ts_main[col].dropna().unique()
        if len(unique_vals) > 0:
             unique_values[col] = unique_vals
    
# Обновляем combination_columns, оставляя только те, для которых есть значения
combination_columns = list(unique_values.keys())

if not combination_columns:
    print("Нет колонок для создания комбинаций или в них нет уникальных значений.")

all_param_combinations = list(product(*[unique_values[col] for col in combination_columns]))
print(f"Количество колонок для комбинаций: {len(combination_columns)}")
print(f"Всего комбинаций для анализа: {len(all_param_combinations)}")

Количество колонок для комбинаций: 4
Всего комбинаций для анализа: 60


In [13]:
# --- Список для сбора всех результатов ---
all_forecast_data_combinations = []
N_FORECAST_YEARS = 5
MIN_OBS_FOR_COMBO_ARIMA = 15

# Execution

In [14]:
# --- Цикл по всем комбинациям ---
for i, combo_values in enumerate(all_param_combinations):
    current_combo_dict = {col: val for col, val in zip(combination_columns, combo_values)}
    combo_str_name_log = "_".join([f"{k}_{v}" for k, v in current_combo_dict.items()])
        
    if (i + 1) % 10 == 0 or i == 0 : # Логируем прогресс
        print(f"\nОбработка комбинации {i+1}/{len(all_param_combinations)}: {combo_str_name_log}")

    # Фильтруем основной DataFrame по текущей комбинации
    df_current_combo_filtered = df_ts_main.copy()
    for col, val in current_combo_dict.items():
        df_current_combo_filtered = df_current_combo_filtered[df_current_combo_filtered[col] == val]
        
    if df_current_combo_filtered.empty:
        print(f"  Пропуск {combo_str_name_log}: нет данных после фильтрации.")
        continue
            
    series_combo_actual = prepare_timeseries_for_combo(df_current_combo_filtered)
        
    result_combo_df = generate_arima_forecast_for_combo(
        series_combo_actual, 
        current_combo_dict,
        n_periods=N_FORECAST_YEARS,
        min_obs_for_arima=MIN_OBS_FOR_COMBO_ARIMA
    )
        
    if result_combo_df is not None and not result_combo_df.empty:
        all_forecast_data_combinations.append(result_combo_df)


Обработка комбинации 1/60: type_D_old_new_Y_duration_F_ppd_type_A
  Пропуск type_D_old_new_Y_duration_U_ppd_type_B: нет данных после фильтрации.

Обработка комбинации 10/60: type_D_old_new_N_duration_L_ppd_type_B
  Пропуск type_D_old_new_N_duration_U_ppd_type_B: нет данных после фильтрации.
  Пропуск type_S_old_new_Y_duration_U_ppd_type_B: нет данных после фильтрации.

Обработка комбинации 20/60: type_S_old_new_N_duration_F_ppd_type_B
  Пропуск type_S_old_new_N_duration_U_ppd_type_B: нет данных после фильтрации.

Обработка комбинации 30/60: type_T_old_new_Y_duration_U_ppd_type_B
  Пропуск type_T_old_new_Y_duration_U_ppd_type_B: нет данных после фильтрации.
  Пропуск type_T_old_new_N_duration_U_ppd_type_B: нет данных после фильтрации.

Обработка комбинации 40/60: type_F_old_new_Y_duration_L_ppd_type_B
  Пропуск type_F_old_new_Y_duration_U_ppd_type_B: нет данных после фильтрации.
  Пропуск type_F_old_new_N_duration_U_ppd_type_B: нет данных после фильтрации.
  Пропуск type_O_old_new_Y_du

In [15]:
# --- Объединение всех данных и сохранение ---
if all_forecast_data_combinations:
    final_combinations_df = pd.concat(all_forecast_data_combinations, ignore_index=True)
        
    output_csv_path_combinations = 'arima_forecasts_combinations_long.csv'
    try:
        # Упорядочим колонки для лучшей читаемости
        cols_order = ['year'] + combination_columns + ['price_value', 'value_type', 'lower_ci', 'upper_ci']
        # Убедимся, что все колонки из cols_order существуют в final_combinations_df
        final_cols_to_save = [col for col in cols_order if col in final_combinations_df.columns]
        # Добавим остальные колонки, если они есть (маловероятно при такой структуре)
        other_cols = [col for col in final_combinations_df.columns if col not in final_cols_to_save]
        final_combinations_df = final_combinations_df[final_cols_to_save + other_cols]


        final_combinations_df.to_csv(output_csv_path_combinations, index=False, float_format='%.0f')
        print(f"\nРезультаты по комбинациям (длинный формат) сохранены в: {output_csv_path_combinations}")
        print(f"Итоговый DataFrame содержит {len(final_combinations_df)} строк.")
        print(final_combinations_df.head())
        print(final_combinations_df.info())
    except Exception as e:
        print(f"\nОшибка при сохранении CSV (комбинации): {e}")


Результаты по комбинациям (длинный формат) сохранены в: arima_forecasts_combinations_long.csv
Итоговый DataFrame содержит 1053 строк.
   year type old_new duration ppd_type    price_value  value_type  lower_ci  \
0  1995    D       Y        F        A  103715.280086  historical       NaN   
1  1996    D       Y        F        A  108653.796590  historical       NaN   
2  1997    D       Y        F        A  114936.093363  historical       NaN   
3  1998    D       Y        F        A  122894.999559  historical       NaN   
4  1999    D       Y        F        A  135631.292484  historical       NaN   

   upper_ci  
0       NaN  
1       NaN  
2       NaN  
3       NaN  
4       NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1053 entries, 0 to 1052
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   year         1053 non-null   int32  
 1   type         1053 non-null   object 
 2   old_new      1053 non-null 